<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Construir un Agente de Matemáticas de Inteligencia Artificial con la Herramienta LangChain

En este laboratorio, aprenderás a crear un agente sencillo con LangChain, lo que permitirá a los agentes de IA realizar tareas específicas. Crearás un conjunto de herramientas matemáticas que permitirá a un agente de IA realizar operaciones aritméticas básicas mediante la interacción en lenguaje natural.

A través de este laboratorio, crearás un agente capaz de comprender y resolver consultas matemáticas como "sumar 25 y 15, y luego multiplicar por 2", descomponiendo operaciones complejas en pasos sencillos.

## 1. Objetivos

Tras completar este laboratorio, podrás:

- Explicar el concepto de herramientas en LangChain
- Crear herramientas personalizadas para tareas específicas
- Desarrollar un agente de IA que pueda usar varias herramientas
- Depurar y mejorar la funcionalidad de las herramientas
- Probar las implementaciones de las herramientas con diferentes entradas

## 2. Configuración


Para este laboratorio, utilizarás las siguientes bibliotecas:

- **`langchain`**: Para crear agentes y herramientas de IA.

- **`langchain.chat_models`**: Para acceder a modelos de lenguaje.

- **`langchain.agents`**: Para crear y gestionar agentes de IA.

### *2.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *2.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [10]:
# from langchain_ibm import ChatWatsonx
from langchain_openai import ChatOpenAI

import re

## 3. Carga del LLM: Elección del Modelo de Lenguaje Adecuado


En este ejemplo, se utilizará `ChatWatsonx` de IBM para cargar un modelo de lenguaje (LLM) que permita interactuar con herramientas. Los modelos de IBM, como Granite 3.2 y Granite 3.3, son muy versátiles y destacan en tareas de razonamiento avanzado.

Sin embargo, otros proveedores ofrecen LLM con diferentes ventajas:

- **OpenAI (GPT-4/GPT-3.5)**: Ideal para versatilidad y razonamiento avanzado.

- **Facebook (Meta, LLaMA)**: De acceso abierto y altamente personalizable para casos de uso especializados.

- **IBM Watsonx Granite**: Perfecto para aplicaciones empresariales con una integración impecable.

- **Anthropic (Claude)**: Centrado en la seguridad, la fiabilidad y la IA ética.

- **Cohere**: Asequible y eficiente para modelos ligeros y específicos para tareas.


Para este proyecto, utilizarás `ChatWatsonx` porque:
- Ofrece una API sencilla para una configuración rápida.

- Admite configuraciones avanzadas como:

- **`temperature`**: Ajusta la aleatoriedad de las respuestas.

- **`max_completion_tokens`**: Limita la longitud de las respuestas.

- Los modelos de IBM son ampliamente reconocidos como de vanguardia para el razonamiento y la conversación de propósito general.

In [2]:
# llm = ChatWatsonx(
#     apikey= "skills-network.eyJleHAiOjB9Cg==",
#     model_id= "ibm/granite-4-h-small",
#     project_id= "skills-network",    
#     url= "https://us-south.ml.cloud.ibm.com"
# )

llm = ChatOpenAI(
    model= 'gpt-4o-mini',
    max_completion_tokens= 512
)

Let's generate a simple response:


In [17]:
respuesta = llm.invoke("¿Qué es la llamada a herramientas en langchain?")
print("\nContenido de la respuesta: ", respuesta.content)


Contenido de la respuesta:  En Langchain, la "llamada a herramientas" se refiere a la capacidad de integrar y utilizar herramientas externas o APIs dentro de una aplicación construida con Langchain. Esto permite que los modelos de lenguaje, como los que se basan en GPT, puedan realizar tareas adicionales que no son posibles solo con su conocimiento interno. 

Por ejemplo, si un modelo necesita realizar búsquedas en internet, acceder a una base de datos, o utilizar funciones específicas como calcular información o generar gráficos, se puede hacer a través de llamadas a herramientas. Estas herramientas pueden ser APIs externas, servicios de terceros, o incluso funciones internas definidas por el desarrollador.

En un flujo de trabajo típico utilizando Langchain, el modelo de lenguaje puede recibir una consulta del usuario, decidir si necesita llamar a una herramienta para obtener más información y luego ejecutar esa llamada, procesar la respuesta y finalmente proporcionar una respuesta 

### *3.1. Aviso Legal Sobre la API*

Este laboratorio utiliza módulos de lenguaje natural (LLM) proporcionados por **IBM watsonx.ai** y **OpenAI**. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente fuera** del entorno JupyterLab de Skills Network, deberá **configurar sus propias claves API**. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

Si ejecuta este laboratorio localmente, deberá configurar sus propias claves API. Este laboratorio utiliza los módulos `ChatOpenAI` y `ChatWatsonx` de `langchain`. Ambas configuraciones se muestran a continuación con sus instrucciones. **Reemplace todas las instancias** de ambos módulos con los módulos completos que se muestran a continuación en todo el laboratorio. **NO** ejecute la celda que se muestra a continuación si no está trabajando localmente, ya que provocará errores.

## 4. Función

En IA, una **herramienta** invoca una **función** o capacidad básica que permite realizar una tarea específica. Imagínelo como una herramienta más en una caja de herramientas: al igual que un martillo, un destornillador o una llave inglesa, una caja de herramientas de IA contiene funciones específicas diseñadas para resolver problemas o realizar tareas.

Al crear herramientas para invocar otras herramientas, es importante tener en cuenta algunos principios clave:

1. **Propósito claro**: Asegúrese de que la herramienta tenga una función bien definida.

2. **Entrada estandarizada**: La herramienta debe aceptar la entrada en un formato predecible y estructurado para facilitar su uso.

3. **Resultados consistentes**: Siempre debe devolver resultados en un formato fácil de procesar o integrar con otros sistemas.

4. **Documentación completa**: Su herramienta debe incluir documentación clara y sencilla que explique su funcionamiento, cómo usarla y cualquier particularidad o limitación.

Recuerda que la documentación no es solo para otros desarrolladores, sino también para que el modelo de lenguaje (LLM) comprenda el propósito de la herramienta y cómo usarla eficazmente.

En este ejemplo, comenzaremos con una herramienta sencilla para sumar números. Cumplirá con la mayoría de los requisitos básicos, pero una limitación clave es que no maneja **errores básicos**, como ignorar entradas no numéricas. Mejorar el manejo de errores hará que la herramienta sea mucho más robusta y esté lista para su uso en situaciones reales.

In [61]:
def agregar_numeros(entradas: str) -> dict:
    """
    Agrega una lista de números proporcionada en un diccionario de entrada o extrae números de una cadena.
    Parámetros:
    - inputs (str):
    cadena que debe contener los números que se pueden extraer y sumar.

    Devuelve:
    - dict: Un diccionario con una sola clave "resultado" que contiene la suma de los números.

    Ejemplo de entrada (diccionario):
    {"numbers": [10, 20, 30]}

    Ejemplo de entrada (cadena):
    "Suma los números 10, 20 y 30."

    Ejemplo de salida:
    {"resultado": 60}
    """

    numbers = [int(x) for x in entradas.replace(",", "").split() if x.isdigit()] # Solo suma números enteros, ignora decimales y otros caracteres.

    resultado = sum(numbers)

    return {"resultado": resultado}

Probar directamente una herramienta permite identificar dónde reside el problema, ya sea en su lógica, en el análisis de la entrada o en el formato de la salida.

Aquí, introducirás la cadena `1 2` y obtendrás la suma.

In [53]:
agregar_numeros("1 2") 

{'resultado': 3.0}

### *4.1. Herramienta*

La clase `Tool` en LangChain actúa como un contenedor estructurado que convierte funciones Python estándar en herramientas compatibles con agentes. Cada herramienta requiere tres componentes clave:
1. Un nombre que la identifique
2. La función que realiza la operación
3. Una descripción que ayude al agente a comprender cuándo usar la herramienta

Mejora de la sección de pruebas:

In [54]:
from langchain_core.tools import Tool

agregar_herramienta = Tool(
        name= "AgregarHerramienta",
        func= agregar_numeros,
        description= "Agregar una lista de números y devolver la suma de ellos.")

print("Objeto de herramienta:", agregar_herramienta)

Objeto de herramienta: name='AgregarHerramienta' description='Agregar una lista de números y devolver la suma de ellos.' func=<function agregar_numeros at 0x771c05116020>


Let's see the parameters of the object:

- **`name`** (*str*):
  - A unique identifier for the tool.

  - **Example**: `"AddTool"`

- **`.invoke`** (*Callable*):
  - The function that the tool wraps.
  - **Example**: `add_numbers`

- **`description`** (*str*):
  - A concise explanation of what the tool does.
  - **Example**: `"Adds a list of numbers and returns the result."`


Veamos los parámetros del objeto:

- **`name`** (*str*):
  - Un identificador único para la herramienta.

  - **Ejemplo**: `"AgregarHerramienta"`

- **`.invoke`** (*Callable*):
  - La función que encapsula la herramienta.

  - **Ejemplo**: `agregar_numeros`

- **`description`** (*str*):
  - Una breve explicación de la función de la herramienta.
  
  - **Ejemplo**: `"Suma una lista de números y devuelve el resultado."`

Estos atributos le permiten inspeccionar el objeto de la herramienta.

In [15]:
# Nombre de la herramienta.
print("Nombre de la herramienta:")
print(agregar_herramienta.name)

# Descripción de la herramienta.
print("Descripción de la herramienta:")
print(agregar_herramienta.description)

# Función de la herramienta
print("Función de la herramienta:")
print(agregar_herramienta.invoke)


Nombre de la herramienta:
AgregarHerramienta
Descripción de la herramienta:
Agregar una lista de números y devolver la suma de ellos.
Función de la herramienta:
<bound method BaseTool.invoke of Tool(name='AgregarHerramienta', description='Agregar una lista de números y devolver la suma de ellos.', func=<function agregar_numeros at 0x771c0555a660>)>


Puedes llamar a la función de la herramienta a través del objeto ```agregar_herramienta```:

In [16]:
print("Llamando a la función de la herramienta:")

test_input = "10 20 30 a b" 

print(agregar_herramienta.invoke(test_input))  # Ejepmlo.

Llamando a la función de la herramienta:
{'resultado': 60}


Las pruebas del objeto de la herramienta garantizan lo siguiente:

1. **La herramienta está configurada correctamente**:
   - Los metadatos (nombre, descripción, etc.) están definidos correctamente y se ajustan a su propósito.

   - La función y el esquema (si corresponde) están configurados correctamente.

2. **La función encapsulada se comporta como se espera**:
   - La función realiza la tarea prevista correctamente.

   - Gestiona adecuadamente los casos límite y las entradas no válidas.

3. **La herramienta se integra sin problemas con los agentes**:
   - La salida de la herramienta coincide con lo que espera el agente.

   - No hay problemas de compatibilidad cuando el agente llama a la herramienta.


### 4.2. Operador `@tool`

Ahora que ya sabes cómo crear una herramienta con la clase `Tool` (usando la interfaz Tool), existe otra forma de crearla: mediante el decorador `@tool`. El método recomendado para crear herramientas es usar el decorador `@tool`. Este decorador está diseñado para simplificar el proceso de creación de herramientas y debería usarse en la mayoría de los casos. Tras definir una función, puedes decorarla con `@tool` para crear una herramienta que implemente la interfaz Tool.

El operador `@tool` convierte funciones en herramientas. Ver más abajo:

In [ ]:
from langchain_core.tools import tool
import re

@tool
def agregar_numeros(entradas: str) -> dict:
    """
    Agrega una lista de números proporcionada en un diccionario de entrada o extrae números de una cadena.
    Parámetros:
    - entradas (str):
    cadena que debe contener los números que se pueden extraer y sumar.

    Devuelve:
    - dict: Un diccionario con una sola clave "resultado" que contiene la suma de los números.

    Ejemplo de entrada (diccionario):
    {"numbers": [10, 20, 30]}

    Ejemplo de entrada (cadena):
    "Suma los números 10, 20 y 30."

    Ejemplo de salida:
    {"resultado": 60}
    """

    numbers = [int(x) for x in entradas.replace(",", "").split() if x.isdigit()]

    resultado = sum(numbers)

    return {"resultado": resultado}

La función anterior ahora actuará como una herramienta. Puede inspeccionar el esquema de la herramienta y otras propiedades utilizando:

In [19]:
print("Nombre: \n", agregar_numeros.name)
print("Descripción: \n", agregar_numeros.description) 
print("Args: \n", agregar_numeros.args) 


Nombre: 
 agregar_numeros
Descripción: 
 Agrega una lista de números proporcionada en un diccionario de entrada o extrae números de una cadena.
Parámetros:
- inputs (str):
cadena que debe contener los números que se pueden extraer y sumar.

Devuelve:
- dict: Un diccionario con una sola clave "resultado" que contiene la suma de los números.

Ejemplo de entrada (diccionario):
{"numbers": [10, 20, 30]}

Ejemplo de entrada (cadena):
"Suma los números 10, 20 y 30."

Ejemplo de salida:
{"resultado": 60}
Args: 
 {'entradas': {'title': 'Entradas', 'type': 'string'}}


Puedes llamar a la herramienta usando el método ```invoke```.

In [21]:
test_input = "Cual es la suma de 10, 20 y 30 " 

print(agregar_numeros.invoke(test_input))  # Ejemplo.

{'resultado': 60}


### *4.3. Usar @tool-StructuredTool*

El decorador @tool crea una StructuredTool con información de esquema extraída de las firmas de las funciones y las cadenas de documentación, como se muestra aquí. Esto ayuda a los desarrolladores de lenguajes de programación a comprender mejor qué entradas espera la herramienta y cómo usarla correctamente. Si bien ambos enfoques funcionan, @tool suele ser la opción preferida para las aplicaciones modernas de LangChain, especialmente con LangGraph y modelos de llamada a funciones.


In [23]:
# Comparación de los dos enfoques.
print("Enfoque de construcción de herramientas:")

print(f"Tiene esquema: {hasattr(agregar_herramienta, 'args_schema')}")
print("\n")

print("@tool Enfoque de decoración:")

print(f"Tiene esquema: {hasattr(agregar_numeros, 'args_schema')}")
print(f"Argumentos: {agregar_numeros.args}")

Enfoque de construcción de herramientas:
Tiene esquema: True


@tool Enfoque de decoración:
Tiene esquema: True
Argumentos: {'entradas': {'title': 'Entradas', 'type': 'string'}}


En este ejemplo, la herramienta tiene dos entradas: una cadena de texto que contiene los números a sumar y una segunda entrada booleana que determina si se deben sumar los valores absolutos de esos números.

In [26]:
from typing import List

@tool
def agregar_numeros_con_opcion(numeros: List[float], absoluto: bool = False) -> float:
    """
    Agrega una lista de números proporcionada como entrada.

    Parámetros:
    - numeros (List[float]): Una lista de números que se sumarán.
    - absoluto (bool): Si es True, utiliza los valores absolutos de los números antes de sumar.

    Devuelve:
    - float: La suma total de los números.
    """
    if absoluto:
        numeros = [abs(n) for n in numeros]

    return sum(numeros)

Comparemos los argumentos de agregar_numeros_con_opcion y agregar_numeros. Ambas son herramientas estructuradas. Ambas incluyen el campo entradas, que es una cadena de texto. Sin embargo, agregar_numeros_con_opcion tiene un par clave-valor adicional: absoluto, un campo booleano con un valor predeterminado de False. Esto significa que agregar_numeros_con_opcion admite un comportamiento opcional (tomar el valor absoluto de los números), mientras que agregar_numeros solo maneja la extracción y suma numérica básica.

In [27]:
print(f"Argumentos: {agregar_numeros.args}")
print(f"Argumentos: {agregar_numeros_con_opcion.args}")

Argumentos: {'entradas': {'title': 'Entradas', 'type': 'string'}}
Argumentos: {'numeros': {'items': {'type': 'number'}, 'title': 'Numeros', 'type': 'array'}, 'absoluto': {'default': False, 'title': 'Absoluto', 'type': 'boolean'}}


Puedes llamar a la herramienta usando un diccionario como entrada, donde cada clave corresponde a un nombre de parámetro y el valor es la entrada para ese parámetro. Por ejemplo, para controlar si los números se suman normalmente o con valores absolutos, establece el indicador absoluto en Falso o Verdadero: Obtendrás -6 y 6, respectivamente.

In [28]:
print(agregar_numeros_con_opcion.invoke({"numeros": [-1.1, -2.1, -3.0], "absoluto": False}))
print(agregar_numeros_con_opcion.invoke({"numeros": [-1.1, -2.1, -3.0], "absoluto": True}))

-6.2
6.2


### *4.4. Tipos de Retorno de Herramientas Mejorados con Tipado Python*

Al crear herramientas, debe especificar con precisión sus valores de retorno. Esto ayuda al agente a comprender y gestionar las diferentes salidas posibles.

La función `sumar_numeros_con_salida_compleja` devuelve un formato de salida más flexible. Devuelve un diccionario que contiene un valor flotante cuando los números se suman correctamente, o un mensaje de error descriptivo en formato de cadena si no se encuentran números o si se produce algún problema durante el procesamiento.

In [29]:
from typing import Dict, Union

@tool
def sumar_numeros_con_salida_compleja(entradas: str) -> Dict[str, Union[float, str]]:
    """
    Extrae y suma todos los números enteros y decimales de la cadena de entrada.

    Parámetros:
    - entradas (str): Una cadena que puede contener valores numéricos.

    Devuelve:
    - dict: Un diccionario con la clave "resultado". Si se encuentran números, el valor es su suma (float).
            Si no se encuentran números o se produce un error, el valor es un mensaje correspondiente (str).

    Ejemplo de entrada:
    "Suma 10, 20.5 y -3."

    Ejemplo de salida:
    {"resultado": 27.5}
    """

    matches = re.findall(r'-?\d+(?:\.\d+)?', entradas)

    if not matches:
        return {"resultado": "No se encontraron números en la entrada."}
    try:
        numeros = [float(num) for num in matches]
        total = sum(numeros)

        return {"resultado": total}
    
    except Exception as e:        
        return {"resultado": f"Error durante la suma: {str(e)}"}

La función `sumar_numeros_desde_texto` devuelve un formato de salida sencillo. Extrae todos los valores enteros de la cadena de entrada, los suma y devuelve el total como un número decimal. Esta función presupone que hay al menos un número válido en la entrada y no gestiona los casos en los que no se encuentran números o en los que puede producirse un error.

In [30]:
@tool
def sumar_numeros_desde_texto(entradas: str) -> float:
    """    
    Agrega una lista de números proporcionada en la cadena de entrada.

    Argumentos:
        entradas: Una cadena que contiene los números que se extraerán y sumarán.

    Devuelve:
        La suma de todos los números encontrados en la entrada.
    """
    # Usar expresiones regulares para extraer números enteros de la cadena de entrada.
    numeros = [int(num) for num in re.findall(r'\d+', entradas)]
    resultado = sum(numeros)

    return resultado

### 4.5. `create_agent`

Al configurar un agente, se conectan herramientas y un LLM para que trabajen juntos sin problemas. El agente utiliza el LLM para comprender qué se necesita hacer y decide qué herramienta usar según la tarea. Aquí hay una descripción general de los componentes clave:

**Relación Entre el Agente y el LLM**
- El **agente** actúa como el responsable de la toma de decisiones, determinando qué herramientas usar y cuándo.

- El **LLM** es el motor de razonamiento. Este:

    - Interpreta la entrada del usuario.

    - Ayuda al agente a tomar decisiones.

    - Genera una respuesta basada en la salida de las herramientas.

Imagina al agente como el administrador que asigna tareas y al LLM como el cerebro que resuelve problemas o delega trabajo.

**Parámetros clave de `create_agent`**

1. **`tools`** - ver arriba

2. **`model`** - ver arriba

3. **`system_prompt`** - Un mensaje que establece el contexto y las instrucciones para el agente. Es crucial para guiar el comportamiento del agente y asegurar que utilice las herramientas de manera efectiva.

Ahora puedes crear un objeto de agente usando `create_agent`.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(model= llm, tools= [agregar_herramienta], system_prompt= "Eres un agente que puede usar herramientas para realizar tareas. Usa la herramienta de agregar números cuando necesites sumar una lista de números.")

Ahora, puedes poner a prueba al agente haciéndole una pregunta.

> Al ejecutar un agente mediante `.invoke()`, es posible que ocasionalmente se encuentre con una situación en la que el código se ejecute indefinidamente, incluso si el LLM ya ha generado una respuesta válida. Esto suele ocurrir cuando el sistema encuentra un `OUTPUT_PARSING_ERROR`, a menudo debido a problemas de formato en la respuesta del LLM.
>
>En estos casos, el agente puede quedar bloqueado en un bucle y no finalizará por sí solo. Si observa que esto sucede, simplemente haga clic en el botón de detener (■) en la barra de herramientas superior para interrumpir la ejecución.

In [37]:
# Usar el agente para realizar una tarea que requiere la herramienta de suma.
respuesta = agente.invoke({'messages': "En 2023, el PIB de EE. UU. fue de aproximadamente 27,72 billones de dólares, mientras que el de Canadá fue de alrededor de 2,14 billones de dólares y el de México de aproximadamente 1,79 billones de dólares. ¿Cuál es el total?"})

In [63]:
respuesta

{'messages': [HumanMessage(content='En 2023, el PIB de EE. UU. fue de aproximadamente 27,72 billones de dólares, mientras que el de Canadá fue de alrededor de 2,14 billones de dólares y el de México de aproximadamente 1,79 billones de dólares. ¿Cuál es el total?', additional_kwargs={}, response_metadata={}, id='96ec2239-1110-491d-b55c-a06288663c0c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 253, 'total_tokens': 281, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0ad65e3318', 'id': 'chatcmpl-DdSdkWBHfEkJ1aAmqqWEdV153vTrL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0ac6-b5fa-7761-896a-bf4121de8

In [62]:
agente.invoke({"messages": "Suma 10, 20, dos y 30"})

{'messages': [HumanMessage(content='Suma 10, 20, dos y 30', additional_kwargs={}, response_metadata={}, id='6c70eaf0-5667-49fd-94b5-e8ee795f896b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 203, 'total_tokens': 233, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0ad65e3318', 'id': 'chatcmpl-DdSnbWp9t9bsiqr18rgpCKRRZTA6k', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0ad0-0a30-7870-98e8-d1d3831387de-0', tool_calls=[{'name': 'agregar_numeros', 'args': {'entradas': 'Suma 10, 20, dos y 30.'}, 'id': 'call_RhIzyOndWUTOochAI872ptvp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

Se le pidió al agente que sumara los números 10, 20, "dos" y 30. El agente notó primero que una de las entradas era la palabra "dos" en lugar de un número, por lo que la convirtió a su forma numérica, que es 2. Después de preparar la lista de números (10, 20, 2 y 30), el agente decidió usar la herramienta `AddTool` para realizar la suma. Le pasó los números a la herramienta, que calculó la suma y devolvió el resultado como 62. Finalmente, el agente proporcionó la respuesta: **62**.

In [66]:
agente_2 = create_agent(model= llm, tools= [sumar_numeros_desde_texto], system_prompt= "Eres un agente que puede usar herramientas para realizar tareas. Usa la herramienta de sumar números desde texto cuando necesites sumar una lista de números extraídos de una cadena de texto.")

agente_2.invoke({"messages": "Suma 10, 20, dos y 30"})

{'messages': [HumanMessage(content='Suma 10, 20, dos y 30', additional_kwargs={}, response_metadata={}, id='26433d82-51de-4b75-8bb2-eb4f59565765'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 139, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_09d726607f', 'id': 'chatcmpl-DdT0brdRxjigbCeFCw6rS4zhPBTCA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0adc-586c-7941-b4c7-70b741963d32-0', tool_calls=[{'name': 'sumar_numeros_desde_texto', 'args': {'entradas': '10, 20, dos, 30'}, 'id': 'call_w1r88OPIQ0kSx5xDRn1eBHEg', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inpu

In [67]:
agente_3 = create_agent(model= llm, tools= [sumar_numeros_con_salida_compleja], system_prompt= "Eres un agente que puede usar herramientas para realizar tareas. Usa la herramienta de sumar números con salida compleja cuando necesites sumar una lista de números extraídos de una cadena de texto y devolver un resultado detallado.")

agente_3.invoke({"messages": "Suma -10, -20 y -30 y dime el resultado con un mensaje detallado."})

{'messages': [HumanMessage(content='Suma -10, -20 y -30 y dime el resultado con un mensaje detallado.', additional_kwargs={}, response_metadata={}, id='a7fa3b75-9a65-4eb6-aef8-b6b185872ab7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 229, 'total_tokens': 263, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_95b7a0331c', 'id': 'chatcmpl-DdT1urw6r8md7D87SjizIxcJS89VH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0add-9019-7b01-92f6-9e55b82d357d-0', tool_calls=[{'name': 'sumar_numeros_con_salida_compleja', 'args': {'entradas': 'Suma -10, -20 y -30.'}, 'id': 'call_FDwMdEqFypkoTqwUYdFzJ4Mh', 'type': 't

In [68]:
agente_4 = create_agent(model= llm, tools = [agregar_numeros_con_opcion], system_prompt= "Eres un agente que puede usar herramientas para realizar tareas. Usa la herramienta de agregar números con opción de absoluto cuando necesites sumar una lista de números extraídos de una cadena de texto y decidir si quieres usar sus valores absolutos o no.")

agente_4.invoke({"messages": "Suma -10, -20 y -30 usando sus valores absolutos."})

{'messages': [HumanMessage(content='Suma -10, -20 y -30 usando sus valores absolutos.', additional_kwargs={}, response_metadata={}, id='f25cca09-a1ae-4ad6-8dfc-33898a5d5049'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 184, 'total_tokens': 216, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1b2df1d8c0', 'id': 'chatcmpl-DdT38VLHyK7jktv4Om5QiJaW4V4xe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0ade-b870-7383-9145-efe642b9f732-0', tool_calls=[{'name': 'agregar_numeros_con_opcion', 'args': {'numeros': [-10, -20, -30], 'absoluto': True}, 'id': 'call_ioqSLcMiXSMqCjU85OKBIDEf', 'type': 'tool_call'}], 

## 5. Orquestación de Múltiples Herramientas con un Agente: Kit de Herramientas Matemáticas

En aplicaciones del mundo real, una sola herramienta suele ser insuficiente para abordar la complejidad y diversidad de las solicitudes de los usuarios. Tareas como el análisis de datos, la realización de cálculos o la recuperación de información específica requieren capacidades especializadas que no pueden cumplirse con una sola función. Al equipar a un agente con múltiples herramientas, cada una adaptada a un propósito distinto, creará un sistema que puede seleccionar y utilizar dinámicamente la herramienta adecuada según la consulta del usuario. Este enfoque mejora la flexibilidad y escalabilidad de la IA, permitiéndole manejar un amplio espectro de tareas con precisión y eficiencia. La orquestación de múltiples herramientas garantiza que el agente pueda gestionar sin problemas flujos de trabajo complejos, lo que lo convierte en un marco esencial para construir sistemas de IA robustos y versátiles.

Para demostrar este concepto, creemos herramientas adicionales, es decir, un conjunto de herramientas matemáticas. Además de la herramienta de suma, ahora crearás herramientas para restar, multiplicar y dividir. Estas herramientas se integrarán en un agente capaz de manejar diversas consultas matemáticas, mostrando cómo múltiples herramientas pueden trabajar juntas dentro de un solo sistema de IA.

### *5.1. Herramienta de resta*


La herramienta de resta está diseñada para tomar una lista de números y devolver el resultado de restar todos los números subsiguientes del primer número. Esta herramienta es particularmente útil para manejar consultas que involucran diferencias, como "¿Qué es 100 menos 20 y luego menos 10?". 

In [72]:
@tool
def restar_numeros(entradas: str) -> dict:
    """
    Extrae números de una cadena, niega el primer número y resta sucesivamente
    los números restantes de la lista.

    Esta función está diseñada para procesar entradas en formato de cadena, donde los números están separados
    por espacios, comas u otros delimitadores. Analiza la cadena, extrae los valores numéricos válidos
    y realiza una resta paso a paso comenzando con el primer número negado.

    Parámetros:
    - entradas (str):
      Una cadena que contiene los números a restar. La cadena puede incluir espacios, comas u
      otros delimitadores entre los números.

    Devuelve:
    - dict:
      Un diccionario que contiene la clave "result" con la diferencia calculada como valor. 
      Si no se encuentran números válidos en la cadena de entrada, el resultado predeterminado es 0.

    Ejemplo de entrada:
    "100, 20, 10"

    Ejemplo de salida:
    {"resultado": -130}

    Notas:
    - Los caracteres no numéricos en la entrada se ignoran.
    - Si la cadena de entrada contiene solo un número válido, el resultado será ese número negado.
    - Admite varios delimitadores (p. ej., espacios, comas), pero no valida los formatos de entrada,

    más allá de extraer valores numéricos.
    """

    # Extraer números de la cadena utilizando una comprensión de lista. Solo se consideran números enteros válidos.
    numeros = [int(num) for num in entradas.replace(",", "").split() if num.isdigit()]

    # si no se encuentran números válidos, devolver un resultado predeterminado de 0.
    if not numeros:
        return {"resultado": 0}

    # Comenzar con el primer número negado.
    result = -1 * numeros[0]

    # restar sucesivamente los números restantes de la lista.
    for num in numeros[1:]:
        result -= num

    return {"resultado": result}

Puede inspeccionar el esquema de la herramienta y otras propiedades utilizando:

In [74]:
print("Nombre: \n", restar_numeros.name)
print("Descripción: \n", restar_numeros.description) 
print("Argumentos: \n", restar_numeros.args) 

Nombre: 
 restar_numeros
Descripción: 
 Extrae números de una cadena, niega el primer número y resta sucesivamente
los números restantes de la lista.

Esta función está diseñada para procesar entradas en formato de cadena, donde los números están separados
por espacios, comas u otros delimitadores. Analiza la cadena, extrae los valores numéricos válidos
y realiza una resta paso a paso comenzando con el primer número negado.

Parámetros:
- entradas (str):
  Una cadena que contiene los números a restar. La cadena puede incluir espacios, comas u
  otros delimitadores entre los números.

Devuelve:
- dict:
  Un diccionario que contiene la clave "result" con la diferencia calculada como valor. 
  Si no se encuentran números válidos en la cadena de entrada, el resultado predeterminado es 0.

Ejemplo de entrada:
"100, 20, 10"

Ejemplo de salida:
{"resultado": -130}

Notas:
- Los caracteres no numéricos en la entrada se ignoran.
- Si la cadena de entrada contiene solo un número válido, el resul

Vamos a usarlo directamente llamando a la función:

In [75]:
print("Llamando a la función de la herramienta:")
test_input = "10 20 30 and cuatro a b" 
print(restar_numeros.invoke(test_input))  # Ejemplo.

Llamando a la función de la herramienta:
{'resultado': -60}


Ahora vamos a crear varias herramientas, empezando por `MultiplicacionTool` y `DivideTool`, definiendo dos funciones: `multiplicar_numeros` y `divide_numeros`. Estas funciones son sencillas: 

`multiplicar_numeros` recibe una lista de números en formato de cadena y devuelve su producto, mientras que `divide_numeros` toma el primer número y lo divide por cada número subsiguiente en secuencia.

In [87]:
# Multiplicacion Tool.
@tool
def multiplicar_numeros(entradas: str) -> dict:
    """
    Extrae números de una cadena y calcula su producto.

    Parámetros:
    - entradas (str): Una cadena que contiene números separados por espacios, comas u otros delimitadores.

    Devuelve:
    - dict: Un diccionario con la clave "resultado" que contiene el producto de los números.

    Ejemplo de entrada:
    "2, 3, 4"

    Ejemplo de salida:
    {"resultado": 24}

    Notas:
    - Si no se encuentra ningún número, el resultado predeterminado es 1 (elemento neutro para la multiplicación).        
    """

    # Extraer números de la cadena utilizando una comprensión de lista. Solo se consideran números enteros válidos.
    numeros = [int(num) for num in entradas.replace(",", "").split() if num.isdigit()]

    # Si no se encuentran números válidos, devolver un resultado predeterminado de 1 (elemento neutro para la multiplicación).
    if not numeros:
        return {"resultado": 1}

    # Calcular el producto de los números.
    resultado = 1
    for num in numeros:
        resultado *= num

    return {"resultado": resultado}

In [78]:
# Division Tool.
@tool
def divide_numeros(entradas: str) -> dict:
    """
    Extrae números de una cadena y calcula el resultado de dividir el primer número 
    por los números subsecuentes en secuencia.

    Parámetros:
    - entradas (str): Una cadena que contiene números separados por espacios, comas o otros delimitadores.

    Devuelve:
    - dict: Un diccionario con la clave "resultado" que contiene el cociente.

    Ejemplo de entrada:
    "100, 5, 2"

    Ejemplo de salida:
    {"resultado": 10.0}

    Notes:
    - Si no se encuentran números, el resultado predeterminado es 0.
    - La división por cero generará un error.
    """
    # Extraer números de la cadena utilizando una comprensión de lista. Solo se consideran números enteros válidos.
    numeros = [int(num) for num in entradas.replace(",", "").split() if num.isdigit()]


    # Si no se encuentran números válidos, devolver un resultado predeterminado de 0.
    if not numeros:
        return {"resultado": 0}

    # Calcular el resultado de dividir el primer número por los números subsecuentes.
    resultado = numeros[0]
    for num in numeros[1:]:
        resultado /= num

    return {"resultado": resultado}

Al probar estas herramientas matemáticas directamente, tenga en cuenta que usar cadenas de texto sin procesar como "2, 3 y cuatro" o "100, 5, dos" provocará un error. Las herramientas están diseñadas para funcionar únicamente con entradas numéricas; no poseen la comprensión del lenguaje natural que ofrece la capa de agente LLM. Para realizar pruebas correctamente, debe usar un valor numérico.

In [88]:
# Prueba de la herramienta de multiplicación.
test_entrada_multiplicacion = "2, 3, and cuatro "
resultado_multiplicacion = multiplicar_numeros.invoke({'entradas': test_entrada_multiplicacion})

print("--- Test MultiplicacionTool ---")
print(f"Input: {test_entrada_multiplicacion}")
print(f"Output: {resultado_multiplicacion}")

--- Test MultiplicacionTool ---
Input: 2, 3, and cuatro 
Output: {'resultado': 6}


In [89]:
# Prueba de la herramienta de división.
test_entrada_division = "100, 5, two"
resultado_divide = divide_numeros.invoke({'entradas': test_entrada_division})
print("--- Test DivisionTool ---")
print(f"Input: {test_entrada_division}")
print(f"Output: {resultado_divide}")

--- Test DivisionTool ---
Input: 100, 5, two
Output: {'resultado': 20.0}


## 6. Construyendo el Agente

Con la implementación de los operadores matemáticos (suma, resta, multiplicación y división), se ha creado un conjunto de herramientas matemáticas sencillo pero funcional. A diferencia de antes, ahora el agente no solo debe seleccionar la herramienta adecuada y procesar la entrada, sino también determinar la operación matemática correcta según la consulta del usuario.

Vamos a crear el objeto agente. Primero, combinemos todas las herramientas en una sola lista:

In [91]:
herramientas = [agregar_numeros, restar_numeros, multiplicar_numeros, divide_numeros]
herramientas    

[<function __main__.agregar_numeros(entradas: str) -> dict>,
 StructuredTool(name='restar_numeros', description='Extrae números de una cadena, niega el primer número y resta sucesivamente\nlos números restantes de la lista.\n\nEsta función está diseñada para procesar entradas en formato de cadena, donde los números están separados\npor espacios, comas u otros delimitadores. Analiza la cadena, extrae los valores numéricos válidos\ny realiza una resta paso a paso comenzando con el primer número negado.\n\nParámetros:\n- entradas (str):\n  Una cadena que contiene los números a restar. La cadena puede incluir espacios, comas u\n  otros delimitadores entre los números.\n\nDevuelve:\n- dict:\n  Un diccionario que contiene la clave "result" con la diferencia calculada como valor. \n  Si no se encuentran números válidos en la cadena de entrada, el resultado predeterminado es 0.\n\nEjemplo de entrada:\n"100, 20, 10"\n\nEjemplo de salida:\n{"resultado": -130}\n\nNotas:\n- Los caracteres no numér

Como antes, crearás el agente utilizando las herramientas y el modelo de lenguaje como datos de entrada.

In [92]:
agente_matematicas = create_agent(model= llm, tools= herramientas, system_prompt= "Eres un agente que puede usar herramientas para realizar tareas matemáticas. Usa la herramienta de agregar números para sumar, la herramienta de restar números para restar, la herramienta de multiplicar números para multiplicar y la herramienta de dividir números para dividir. Elige la herramienta adecuada según la consulta del usuario.")

In [94]:
respuesta = agente_matematicas.invoke({
    "messages": [("human", "¿Cuanto es 25 dividido por 4?")]
})
ultima_respuesta = respuesta["messages"][-1].content

# Obtenemos la respuesta final del agente.
print(respuesta)
print('\n')
print(ultima_respuesta)

{'messages': [HumanMessage(content='¿Cuanto es 25 dividido por 4?', additional_kwargs={}, response_metadata={}, id='75775b70-79e1-49cd-a53f-e22051de5279'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 795, 'total_tokens': 815, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e2d886d409', 'id': 'chatcmpl-DdTYA4xcWPlxdyDZHS0aF5rbRtPUK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0afc-179c-7780-903f-d8d3e0bb1297-0', tool_calls=[{'name': 'divide_numeros', 'args': {'entradas': '25, 4'}, 'id': 'call_z0YyjL5PxMncKs9hSTZjieaw', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 795,

In [96]:
repuesta_2 = agente_matematicas.invoke({
    "messages": [("human", "Resta 100, 20 y 10.")]
})
ultima_respuesta_2 = repuesta_2["messages"][-1].content

# Obtenemos la respuesta final del agente.
print(ultima_respuesta_2)

La resta de 100, 20 y 10 es -130.


Cuando el agente intenta restar 20 y 10 de 100, ocurre algo inesperado. La herramienta llamada `RestarTool` funciona de manera diferente a lo que el agente esperaba. Al escribir "100, 20, 10", en lugar de dar 70 como se esperaría, da -130. Esto sucede porque la calculadora especial primero convierte 100 en -100 y luego resta los otros números.

```La confusión```

El agente espera que la función funcione como una operación matemática normal (100 - 20 - 10 = 70). Cuando el agente intenta solucionar esto dividiendo el problema en pasos más pequeños, sigue obteniendo resultados inesperados porque la calculadora continúa usando sus reglas especiales.

```Atascado```

El agente sigue intentando el mismo método repetidamente, sin darse cuenta de por qué no funciona. Finalmente, se queda sin tiempo sin resolver el problema.

Antes de solucionar el problema, probemos las otras herramientas.

In [97]:
print("\n--- Prueba MultiplicarTool ---")
respuesta = agente_matematicas.invoke({
    "messages": [("human", "Mutiplica 2, 3 y cuatro.")]
})
print("Respuesta del Agente:", respuesta["messages"][-1].content)

print("\n--- Prueba DividirTool ---")
respuesta = agente_matematicas.invoke({
    "messages": [("human", "Divide 100 entre 5 y luego entre 2.")]
})
print("Respuesta del Agente:", respuesta["messages"][-1].content)


--- Prueba MultiplicarTool ---
Respuesta del Agente: El producto de 2, 3 y 4 es 24.

--- Prueba DividirTool ---
Respuesta del Agente: El resultado de dividir 100 entre 5 y luego entre 2 es 10.0.


Ahora vamos a modificar la herramienta `RestarTool` para que reste los números directamente (sin negar el primer número). Esto alinea el comportamiento de la herramienta con la aritmética estándar y las expectativas del agente.

In [98]:
@tool
def nuevo_restar_numeros(entradas: str) -> dict:
    """
    Extrae números de una cadena y realiza la resta secuencialmente, comenzando por el primer número.

    Esta función está diseñada para procesar entradas en formato de cadena, donde los números pueden estar separados por espacios,

    comas u otros delimitadores. Analiza la cadena de entrada, extrae los valores numéricos y calcula
    el resultado restando cada número subsiguiente del primero. entradas[0]-entradas[1]-entradas[2].

    Parámetros:
    - entradas (str):
      Una cadena que contiene los números a restar. La cadena puede incluir espacios, comas u otros
      delimitadores entre los números.

    Devuelve:
    - dict:
      Un diccionario que contiene la clave "resultado" con la diferencia calculada como valor. 
      Si no se encuentran números válidos en la cadena de entrada, el resultado predeterminado es 0.

    Ejemplo de uso:
    - Entrada: "100, 20, 10"
    - Salida: {"resultado": 70}

    Limitaciones:
    - La función no admite números con formato decimal u otras representaciones no enteras.
    """
    # Extraer números de la cadena utilizando una comprensión de lista. Solo se consideran números enteros válidos.
    numbers = [int(num) for num in entradas.replace(",", "").split() if num.isdigit()]

    # Si no se encuentran números válidos, devolver un resultado predeterminado de 0.
    if not numbers:
        return {"resultado": 0}

    # Comenzar con el primer número.
    resultado = numbers[0]

    # Restar todos los números subsiguientes.
    for num in numbers[1:]:
        resultado -= num

    return {"resultado": resultado}

## 7. ​​Nota: Nomenclatura de Herramientas al Demostrar Diferentes Enfoques

En este laboratorio, se muestran intencionalmente dos maneras diferentes de crear la misma herramienta matemática (suma):

1. Usando el constructor `Tool()` (`agregar_herramienta`)
2. Usando el decorador `@tool` (`agregar_numeros`)

Esto se hace para comparar diferentes métodos de creación de herramientas de LangChain con fines educativos. En una aplicación de producción, normalmente se elige un enfoque consistente en lugar de tener herramientas duplicadas para la misma funcionalidad.

Al crear agentes reales, las herramientas duplicadas con funciones similares confundirán al LLM, ya que no sabrá cuál elegir. Siempre use herramientas únicas con propósitos claramente diferenciados en el código de producción.

A continuación, cree un nuevo agente, asegurándose de que incorpore la herramienta de resta actualizada.

In [99]:
tools_updated = [agregar_numeros, nuevo_restar_numeros, multiplicar_numeros, divide_numeros]

# Crear el agente con todas las herramientas.
nuevo_agente_matematicas = create_agent(
    model= llm,
    tools= tools_updated,
    # Opcionalmente, puedes actualizar el mensaje del sistema para reflejar la nueva herramienta de resta.
    system_prompt= "Eres un agente que puede usar herramientas para realizar tareas matemáticas. Usa la herramienta de agregar números para sumar, la herramienta de restar números para restar, la herramienta de multiplicar números para multiplicar y la herramienta de dividir números para dividir. Elige la herramienta adecuada según la consulta del usuario."
)

print("Agente", nuevo_agente_matematicas)

Agente <langgraph.graph.state.CompiledStateGraph object at 0x771c047b40e0>


Ahora, crearás un diccionario de Python para probar varios casos de uso de tu agente. Probar el agente es importante en sí mismo, ya que ayuda a garantizar que funcione correctamente en diferentes situaciones. Automatizar los casos de prueba facilita este proceso y ayuda a detectar errores antes de que se conviertan en un problema. Un buen conjunto de pruebas verifica cómo el agente maneja diferentes entradas, incluyendo casos complejos como la división por cero, el trabajo con números grandes y el manejo de decimales. También puedes probar cómo el agente maneja operaciones mixtas, como la suma y la multiplicación.

In [104]:
# Casos de prueba para el nuevo agente con la herramienta de resta actualizada.
casos_prueba = [
    {
        "consulta": "Resta 100, 20 y 10.",
        "expectativa": {"resultado": 70},
        "descripcion": "Prueba de la herramienta de resta con resta secuencial."
    },
    {
        "consulta": "Multiplica 2, 3 y 4.",
        "expectativa": {"resultado": 24},
        "descripcion": "Prueba de la herramienta de multiplicación para una lista de números."
    },
    {
        "consulta": "Divide 100 entre 5 y luego entre 2.",
        "expectativa": {"resultado": 10.0},
        "descripcion": "Prueba de la herramienta de división con división secuencial."
    },
    {
        "consulta": "Resta 50 de 20.",
        "expectativa": {"resultado": -30},
        "descripcion": "Prueba de la herramienta de resta con resultados negativos."
    }

]

Este código extrae el resultado real del cálculo de la estructura de respuesta del agente. A diferencia de una invocación directa de una herramienta que devuelve un diccionario simple, los agentes de LangGraph devuelven una estructura compleja que contiene todo el historial de la conversación como una lista de mensajes. Para encontrar el resultado del cálculo, debe localizar el ToolMessage específico en esta lista (identificado por su nombre, que coincide con el de una de las herramientas matemáticas) y, a continuación, analizar su contenido, que contiene el resultado real como una cadena JSON. Este enfoque es necesario porque el resultado no es accesible directamente desde el objeto de respuesta, sino que está anidado dentro del historial de mensajes, lo que requiere navegar por los mensajes para encontrar y extraer los datos relevantes para compararlos con los valores esperados.

In [106]:
tareas_correctas = []

# Prueba de cada caso utilizando el nuevo agente.
for index, prueba in enumerate(casos_prueba, start= 1):
    consulta = prueba["consulta"]
    expectativa_resultado = prueba["expectativa"]["resultado"]  # Extraer solo el valor.
    
    print(f"\n--- Caso de Prueba {index}: {prueba['descripcion']} ---")
    print(f"Consulta: {consulta}")
    
    # Formatear correctamente la entrada para el agente.
    respuesta = nuevo_agente_matematicas.invoke({"messages": [("human", consulta)]})
    
    # Encontrar el ToolMessage específico en la respuesta del agente para extraer el resultado de la herramienta.
    tool_message = None
    for msg in respuesta["messages"]:        
        if hasattr(msg, 'name') and msg.name in ['agregar_numeros', 'nuevo_restar_numeros', 'multiplicar_numeros', 'divide_numeros']:
            tool_message = msg
            break
    
    if tool_message:
        # Formatear el resultado de la herramienta para compararlo con el valor esperado.
        import json

        resultado_herramienta = json.loads(tool_message.content)["resultado"]
        print(f"Resultado de la herramienta: {resultado_herramienta}")
        print(f"Resultado esperado: {expectativa_resultado}")
        
        if resultado_herramienta == expectativa_resultado:
            print(f"✅ Paso la prueba: {prueba['descripcion']}")
            tareas_correctas.append(prueba["descripcion"])
        else:
            print(f"❌ Prueba fallida: {prueba['descripcion']}")
    else:
        print("❌ No se llamó a ninguna herramienta por parte del agente")

print("\nTareas correctamente pasadas:", tareas_correctas)


--- Caso de Prueba 1: Prueba de la herramienta de resta con resta secuencial. ---
Consulta: Resta 100, 20 y 10.
Resultado de la herramienta: 70
Resultado esperado: 70
✅ Paso la prueba: Prueba de la herramienta de resta con resta secuencial.

--- Caso de Prueba 2: Prueba de la herramienta de multiplicación para una lista de números. ---
Consulta: Multiplica 2, 3 y 4.
Resultado de la herramienta: 24
Resultado esperado: 24
✅ Paso la prueba: Prueba de la herramienta de multiplicación para una lista de números.

--- Caso de Prueba 3: Prueba de la herramienta de división con división secuencial. ---
Consulta: Divide 100 entre 5 y luego entre 2.
Resultado de la herramienta: 10.0
Resultado esperado: 10.0
✅ Paso la prueba: Prueba de la herramienta de división con división secuencial.

--- Caso de Prueba 4: Prueba de la herramienta de resta con resultados negativos. ---
Consulta: Resta 50 de 20.
Resultado de la herramienta: 30
Resultado esperado: -30
❌ Prueba fallida: Prueba de la herramienta d

Las funciones actuales se beneficiarían de una mejor gestión de errores y validación de entrada. Las funciones `agregar_numeros`, `nuevo_restar_numeros`, `multiplicar_numeros` y `divide_numeros` deberían actualizarse para manejar números decimales mediante conversión a coma flotante, validar las entradas de forma más estricta y proporcionar mensajes de error claros para casos límite. 

Por ejemplo, `divide_numeros` debería comprobar explícitamente la división por cero, y todas las funciones deberían manejar correctamente las entradas no numéricas como "dos" o "cien". Los casos de prueba deberían ampliarse más allá de las operaciones básicas para incluir casos límite como la división por cero, entradas vacías y entradas mixtas numéricas/de texto (p. ej., "dividir cien entre 5"). 

También se debería considerar añadir pruebas para números decimales (p. ej., "multiplicar 3,5 por 2") y operaciones secuenciales (p. ej., "multiplicar 10 por 2 y luego sumar 5"). Este enfoque de pruebas integral garantiza que el agente pueda manejar una amplia gama de consultas matemáticas del mundo real.

In [107]:
nuevo_agente_matematicas.invoke({"messages": [("human", "Multiplica 20 * 30 y luego restale 50.")]})

{'messages': [HumanMessage(content='Multiplica 20 * 30 y luego restale 50.', additional_kwargs={}, response_metadata={}, id='5e2d41f6-1411-4625-87ed-abddbec7016f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 758, 'total_tokens': 816, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e2d886d409', 'id': 'chatcmpl-DdU2OPi6gH2ZHmcgqpdmsDWhDD1VL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0b18-b000-7c21-bb37-088c5e760281-0', tool_calls=[{'name': 'multiplicar_numeros', 'args': {'entradas': '20, 30'}, 'id': 'call_G1vvsWkQQdjdKdmYzJ9Zslp3', 'type': 'tool_call'}, {'name': 'nuevo_restar_numeros', 'args': {'

## 8. Explorando las Herramientas Integradas de LangChain

Si bien crear herramientas personalizadas es muy útil, LangChain ofrece un amplio ecosistema de herramientas prediseñadas que resuelven tareas comunes de forma inmediata. Estas herramientas abstraen los detalles complejos de implementación (llamadas a la API, análisis de entrada, manejo de errores) y permiten centrarse en la creación rápida de agentes robustos.

### *8.1. Herramientas Integradas Populares*





Aquí tienes algunas herramientas muy utilizadas de `langchain_community.tools`:

| Nombre de la herramienta| Descripción                                                                 |
|-------------------------|-----------------------------------------------------------------------------|
| `WikipediaQueryRun`     | Busca información verídica en Wikipedia.                                    |
| `GoogleSearchRun`       | Realiza búsquedas web usando la API de Google.                              |
| `PythonREPLTool`        | Ejecuta código Python en un entorno seguro.                                 |
| `OpenWeatherMapQueryRun`| Obtiene datos meteorológicos en tiempo real.                                |
| `YouTubeSearchTool`     | Busca vídeos de YouTube.                                                    |


### *8.2. Ejemplo: Uso de la Herramienta de Wikipedia*

Vamos a mejorar el agente matemático con acceso a Wikipedia para que pueda responder preguntas que requieran contexto verdadero.

Ahora, comenzarás creando una herramienta de búsqueda en Wikipedia usando el operador `@tool`. Esta herramienta permitirá al agente obtener información objetiva de Wikipedia cuando sea necesario.

In [110]:
from langchain_community.utilities import WikipediaAPIWrapper

# Crear una herramienta de búsqueda en Wikipedia usando el operador @tool.
@tool
def busqueda_wikipedia(consulta: str) -> str:
    """Buscar en Wikipedia información factual sobre un tema.
    
    Parametros:
    - consulta (str): El tema o pregunta para buscar en Wikipedia
    
    Devuelve:
    - str: Un resumen de la información relevante de Wikipedia
    """
    wiki = WikipediaAPIWrapper(lang= "es")  # Puedes especificar el idioma si lo deseas, por ejemplo, "es" para español.

    return wiki.run(consulta)

In [113]:
busqueda_wikipedia.invoke("¿Que es un llamado a herramientas en LangChain IA?")

'Page: Microsoft Dynamics 365\nSummary: Microsoft Dynamics 365 es un conjunto integrado de aplicaciones de planificación de recursos empresariales (ERP) y gestión de relaciones con el cliente (CRM) ofrecidas por Microsoft.\u200b Su producto estrella, Dynamics GP, fue fundado en 1981.\n\nPage: Unity Technologies\nSummary: Unity Software Inc. (también conocida como Unity Technologies) es una compañía americana de desarrollo de software para la creación de videojuegos, con sede en San Francisco. Se fundó en 2004 bajo el nombre de Over the Edge Entertainment (OTEE), en Dinamarca. Cambiando su nombre en 2007. Unity Technologies es conocida por crear Unity, un motor de videojuegos.\nEl motor se puede utilizar para crear juegos tridimensionales (3D) y bidimensionales (2D), así como simulaciones interactivas y otras experiencias. El motor ha sido adoptado por industrias ajenas a los videojuegos, como el cine, la automoción, la arquitectura, la ingeniería, la construcción y la milicia.\n\nPage:

Ahora, **crearás una lista de herramientas disponibles** (tanto herramientas matemáticas personalizadas como la herramienta de Wikipedia `busqueda_wikipedia`) y luego **inicializarás un agente** que pueda usar estas herramientas para resolver problemas. Este agente combinará:

- Herramientas matemáticas personalizadas (`agregar_numeros`, `nuevo_restar_numeros`, etc.) para operaciones aritméticas.

- Una herramienta integrada (`wiki_tool`, por ejemplo, para búsquedas en Wikipedia) para funcionalidades adicionales.

Al combinar estas herramientas, el agente puede gestionar **cálculos matemáticos** (por ejemplo, suma, resta) y **consultas de información** (por ejemplo, obtener datos de Wikipedia), según la solicitud del usuario.

In [114]:
# Actualiza tu lista de herramientas para incluir la herramienta de Wikipedia.
actualizar_herramientas = [agregar_numeros, nuevo_restar_numeros, multiplicar_numeros, divide_numeros, busqueda_wikipedia]

# Crear el agente con todas las herramientas incluyendo Wikipedia.
nuevo_agente_matematico_wikipediko = create_agent(
    model= llm,
    tools= actualizar_herramientas,
    system_prompt= "Eres un agente que puede usar herramientas para realizar tareas. Usa la herramienta de agregar números para sumar, la herramienta de restar números para restar, la herramienta de multiplicar números para multiplicar, la herramienta de dividir números para dividir y la herramienta de búsqueda en Wikipedia para obtener información factual. Elige la herramienta adecuada según la consulta del usuario."
)

Ahora, **le harás al agente una pregunta de varios pasos** que requiere:
1. **Búsqueda en línea** (usando `busqueda_wikipedia` u otra herramienta integrada) para obtener datos reales.

2. **Cálculo matemático** (usando `multiplicar_numeros`) para procesar los datos obtenidos.

In [115]:
consulta = "¿Cúal es la población de Canadá? Multiplicala por 0.75"

respuesta = nuevo_agente_matematico_wikipediko.invoke({"messages": [("human", consulta)]})

print("\nSecuencia de Mensajes:")
for i, msg in enumerate(respuesta["messages"]):
    print(f"\n--- Mensaje {i+1} ---")
    print(f"Tipo: {type(msg).__name__}")

    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")

    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
        
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")


Secuencia de Mensajes:

--- Mensaje 1 ---
Tipo: HumanMessage
Content: ¿Cúal es la población de Canadá? Multiplicala por 0.75
Name: None

--- Mensaje 2 ---
Tipo: AIMessage
Content: 
Name: None
Tool calls: [{'name': 'busqueda_wikipedia', 'args': {'consulta': 'población de Canadá'}, 'id': 'call_CXVmO7bNn8Seb3tRM53U3kOc', 'type': 'tool_call'}]

--- Mensaje 3 ---
Tipo: ToolMessage
Content: Page: Población de Canadá
Summary: Canadá ocupa el puesto 37 por población, que comprende aproximadamente el 0.5% del total mundial,​ con más de 39 millones de canadienses al 2022.​ Sin embargo, al ser el cuarto país más grande por área terrestre (el segundo más grande por área total), la gran mayoría del país está escasamente habitada, con la mayor parte de su población al sur del paralelo 55 norte y poco más del 60 por ciento de los canadienses viven en solo dos provincias: Ontario y Quebec. Aunque la densidad de población de Canadá es baja, muchas regiones del sur, como el corredor de la ciudad de Que

**Cómo funciona**:
1. El agente primero usa `busqueda_wikipedia` para encontrar la población de Canadá.

2. Extrae el valor numérico de la respuesta de Wikipedia.

3. Usa `multiplicar_numeros` para calcular el 75 % de la población.

4. Devuelve el resultado final con contexto.

Para obtener una lista completa de las herramientas disponibles, consulte la [Documentación de herramientas de LangChain](https://python.langchain.com/docs/integrations/tools/).

## 9. Ejercicio: Crea una Herramienta Eléctrica para Calcular Exponentes

### *9.1. Objetivo*

En este ejercicio, crearás una herramienta personalizada que calcula la potencia de un número (por ejemplo, \( x^y \)). A continuación, integrarás esta herramienta en un agente y probarás su funcionamiento.

### *9.2. Crear la Herramienta Potencia*

1. **Definir la función de la herramienta**:

- Crea una función de Python llamada `calcular_potencia` que reciba como entrada una cadena de texto. Esta cadena contendrá dos números: la base (\( x \)) y el exponente (\( y \)).

- La función deberá extraer los números, calcular \( x^y \) y devolver el resultado como un diccionario con la clave `"resultado"`.

In [117]:
@tool
def calcular_potencia(base: float, exponente: float) -> Dict[str, float]:
    """
    Calcula la potencia de un número dado una base y un exponente.

    Parámetros:
    - base (float): El número base que se va a elevar a la potencia.
    - exponente (float): El número al que se eleva la base.

    Devuelve:
    - Dict[str, float]: Un diccionario con la base, exponente y resultado.

    Ejemplo de uso:
    - Entrada: base=2, exponente=3
    - Salida: {"resultado": 8.0}

    Notas:
    - La función utiliza la operación de potencia estándar, por lo que también puede manejar exponentes negativos y decimales.    
    """

    return {"resultado": base ** exponente}

In [118]:
agente_pontencia = create_agent(
    model= llm,
    tools= [calcular_potencia],
    system_prompt= "Eres un agente que puede usar herramientas para realizar tareas matemáticas. Usa la herramienta de calcular potencia para elevar un número a una potencia dada. Elige la herramienta adecuada según la consulta del usuario."
)

In [124]:
agente_pontencia.invoke({'messages': "Calcula la potencia de 2 elevado a la 2 y luego volver a elevar el resultado a la 3."})

{'messages': [HumanMessage(content='Calcula la potencia de 2 elevado a la 2 y luego volver a elevar el resultado a la 3.', additional_kwargs={}, response_metadata={}, id='70944e13-d65a-4623-bf61-b7f102b2757e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 237, 'total_tokens': 297, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9bab15d2ac', 'id': 'chatcmpl-DdUyBhjajGtkVyIf9w5I9fuUo8y2C', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e0b4f-5c62-7850-b8cc-2ab5656382a4-0', tool_calls=[{'name': 'calcular_potencia', 'args': {'base': 2, 'exponente': 2}, 'id': 'call_XE4EWnP3k1oUyhIGotrkJ0VT', 'type': 'tool_ca

Copyright © IBM Corporation. All rights reserved.
